In [0]:
# Verify orders_raw is accessible
df = spark.table("intelligent_etl.default.orders_raw")
print(f"Total rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.show(5, truncate=False)

Total rows: 10,000
Columns: 13
+------------------------------------+-----------+----------+-------------+-----------+--------+----------+------------+-------------------+------+--------------+----------+------------+
|order_id                            |customer_id|product_id|product_name |category   |quantity|unit_price|total_amount|order_date         |region|payment_method|status    |discount_pct|
+------------------------------------+-----------+----------+-------------+-----------+--------+----------+------------+-------------------+------+--------------+----------+------------+
|4c3fd1de-d34a-46d2-a515-52def7cd7941|CUST9774   |P001      |Laptop       |Electronics|1       |106992.75 |106992.75   |2024-09-27 00:00:00|South |Debit Card    |Cancelled |0.062       |
|666484d7-2065-4dc7-867a-00474ac40f8a|CUST3514   |P007      |Backpack     |Accessories|1       |1684.29   |1684.29     |2024-01-27 00:00:00|South |Debit Card    |Returned  |0.175       |
|6132f5cd-69c7-4623-a561-4efdfa0e3

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Config
CATALOG = "intelligent_etl"
SCHEMA  = "default"
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {RUN_ID}")

Run ID: 20260513_131809


In [0]:
# Read raw source table
raw_df = spark.table(f"{CATALOG}.{SCHEMA}.orders_raw")

# Add Bronze metadata columns
bronze_df = (
    raw_df
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_run_id",       F.lit(RUN_ID))
    .withColumn("_source",       F.lit("orders_raw_csv"))
)

print(f"Bronze rows: {bronze_df.count():,}")
print(f"Columns: {bronze_df.columns}")
bronze_df.show(3, truncate=True)

Bronze rows: 10,000
Columns: ['order_id', 'customer_id', 'product_id', 'product_name', 'category', 'quantity', 'unit_price', 'total_amount', 'order_date', 'region', 'payment_method', 'status', 'discount_pct', '_ingestion_ts', '_run_id', '_source']
+--------------------+-----------+----------+------------+-----------+--------+----------+------------+-------------------+------+--------------+---------+------------+--------------------+---------------+--------------+
|            order_id|customer_id|product_id|product_name|   category|quantity|unit_price|total_amount|         order_date|region|payment_method|   status|discount_pct|       _ingestion_ts|        _run_id|       _source|
+--------------------+-----------+----------+------------+-----------+--------+----------+------------+-------------------+------+--------------+---------+------------+--------------------+---------------+--------------+
|4c3fd1de-d34a-46d...|   CUST9774|      P001|      Laptop|Electronics|       1| 106992.75

In [0]:
# Write to Bronze Delta table
(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.bronze_orders")
)

print("Bronze table written successfully")

# Verify
spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders").count()

Bronze table written successfully


10000

In [0]:
# Check Bronze table
bronze = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")

print(f"Rows:    {bronze.count():,}")
print(f"Columns: {len(bronze.columns)}")
print("\nMetadata columns added:")
bronze.select("order_id", "_ingestion_ts", "_run_id", "_source").show(3)

Rows:    10,000
Columns: 16

Metadata columns added:
+--------------------+--------------------+---------------+--------------+
|            order_id|       _ingestion_ts|        _run_id|       _source|
+--------------------+--------------------+---------------+--------------+
|4c3fd1de-d34a-46d...|2026-05-13 13:18:...|20260513_131809|orders_raw_csv|
|666484d7-2065-4dc...|2026-05-13 13:18:...|20260513_131809|orders_raw_csv|
|6132f5cd-69c7-462...|2026-05-13 13:18:...|20260513_131809|orders_raw_csv|
+--------------------+--------------------+---------------+--------------+
only showing top 3 rows
